In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from tensorflow.keras.models import load_model
import os
import sys

# Add parent path if needed
sys.path.append('..')

print("✅ Imports loaded")

2026-05-10 17:48:54.938405: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


✅ Imports loaded


In [2]:
# Paths to model files
repo_path = '../Stock-Price-Movement-Prediction'
model_path = os.path.join(repo_path, 'best_enhanced_model.keras')
scaler_path = os.path.join(repo_path, 'artifacts', 'enhanced', 'feature_scaler_enhanced.pkl')

# Load
model = load_model(model_path)
scaler = joblib.load(scaler_path)

print("✅ Model and scaler loaded")
print(f"Model input shape: {model.input_shape}")
print(f"Model output shape: {model.output_shape}")
print(f"Scaler expects: {scaler.n_features_in_} features")

✅ Model and scaler loaded
Model input shape: (None, 120, 24)
Model output shape: (None, 1)
Scaler expects: 24 features


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.7.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [3]:
# Load your simulated paths
normal_paths = np.load('../data/simulated/mc_returns_normal.npy')
crisis_2008_paths = np.load('../data/simulated/mc_returns_crisis_2008.npy')
crisis_covid_paths = np.load('../data/simulated/mc_returns_crisis_covid.npy')
synthetic_paths = np.load('../data/simulated/mc_returns_synthetic_extreme.npy')

print("✅ Monte Carlo paths loaded")
print(f"Normal paths: {normal_paths.shape}")      # (10000, 252)
print(f"2008 paths:   {crisis_2008_paths.shape}")
print(f"COVID paths:  {crisis_covid_paths.shape}")
print(f"Synthetic:    {synthetic_paths.shape}")

✅ Monte Carlo paths loaded
Normal paths: (10000, 252)
2008 paths:   (10000, 252)
COVID paths:  (10000, 252)
Synthetic:    (10000, 252)


In [8]:
def create_24_features(returns_paths, lookback=120):
    """
    Convert Monte Carlo returns to 24-feature format.
    """
    n_paths, n_days = returns_paths.shape
    n_features = 24
    
    # Take last 'lookback' days
    returns_window = returns_paths[:, -lookback:]  # (n_paths, 120)
    
    # Initialize features
    features = np.zeros((n_paths, lookback, n_features))
    
    # Feature 0: Raw returns
    features[:, :, 0] = returns_window
    
    # Features 1-23: Simplified approach
    for i in range(1, n_features):
        decay = 0.95 ** i
        features[:, :, i] = returns_window * decay
    
    return features

print("✅ Adapter function ready")

✅ Adapter function ready


In [9]:
def price_to_direction_accuracy(predictions, returns_paths, lookback=120):
    """
    Convert price predictions to directional accuracy.
    """
    n_paths = returns_paths.shape[0]
    
    # Build price series from returns (starting at 100)
    prices = 100 * np.cumprod(1 + returns_paths, axis=1)
    
    # Get the last price BEFORE the prediction day
    last_price = prices[:, -(lookback+1)]
    
    # Predicted direction: 1 if predicted price > last price
    pred_prices = predictions.flatten()
    pred_direction = (pred_prices > last_price).astype(int)
    
    # Actual direction: 1 if NEXT day's return > 0
    next_return = returns_paths[:, -(lookback+1)]
    actual_direction = (next_return > 0).astype(int)
    
    # Calculate accuracy
    correct = (pred_direction == actual_direction).sum()
    accuracy = correct / n_paths * 100
    
    return accuracy, pred_direction, actual_direction

print("✅ Accuracy function ready")

✅ Accuracy function ready


In [10]:
# Test on first 100 paths
print("Generating test predictions...")
test_batch = normal_paths[:100]
test_features = create_24_features(test_batch, lookback=120)

# Reshape, scale, predict
test_2d = test_features.reshape(-1, 24)
test_scaled = scaler.transform(test_2d)
test_scaled_3d = test_scaled.reshape(-1, 120, 24)
test_preds = model.predict(test_scaled_3d, verbose=0)

print(f"✅ Predictions generated: {test_preds.shape}")
print(f"Prediction range: {test_preds.min():.2f} to {test_preds.max():.2f}")

Generating test predictions...
✅ Predictions generated: (100, 1)
Prediction range: 13587.28 to 13612.94


In [11]:
# Calculate directional accuracy
test_acc, _, _ = price_to_direction_accuracy(test_preds, normal_paths[:100], lookback=120)
print(f"\n{'='*50}")
print(f"DIRECTIONAL ACCURACY ON 100 PATHS")
print(f"{'='*50}")
print(f"Accuracy: {test_acc:.2f}%")
print(f"{'='*50}")

if test_acc > 55:
    print("✅ Good! Model is usable. Proceed to full 10,000 paths.")
elif test_acc > 50:
    print("⚠️ Marginal. Model is barely better than random. Consider training your own.")
else:
    print("❌ Model is random (≤50%). Need to train a different model.")


DIRECTIONAL ACCURACY ON 100 PATHS
Accuracy: 54.00%
⚠️ Marginal. Model is barely better than random. Consider training your own.
